# ChakraModel V5 Strict Verification - True Anti-Fabrication Harness

This notebook copies your chosen dataset subset to the working directory, plants the Anti-Fabrication canaries, and runs the hardened `verify_kaggle.py` script via `harness_core`.

**Before running:**
1. Make sure you have added the `ChakraModel_Kaggle_Code` and `Anti-Fabrication Toolkit` datasets.
2. Ensure you run this using **Save Version -> Save & Run All (Commit)**.

In [1]:
!pip install -q ultralytics timm opencv-python-headless pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.6 MB/s eta 0:00:00


In [2]:
import os
import sys
import shutil
import glob
import json
import re

###################################################################
# CONFIGURATION
###################################################################

# The EXACT path for the shared dataset on your new account
DATASET_ROOT = '/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI'

# The CSV and JSON Card will use this name
VERIFY_DATASET_NAME = "PolypDB_BKAI"

WORKING_DATASET = '/kaggle/working/dataset_copy'

if os.path.exists(WORKING_DATASET):
    shutil.rmtree(WORKING_DATASET)
    
print(f"Copying files from {DATASET_ROOT} ...")
shutil.copytree(DATASET_ROOT, WORKING_DATASET)
print(f"Dataset securely copied to {WORKING_DATASET}")

Copying files from /kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI ...
Dataset securely copied to /kaggle/working/dataset_copy


In [3]:
import os

def print_directory_tree(startpath, max_depth=4):
    print(f"🗺️ Directory tree for: {startpath}")
    print("=" * 50)
    
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        
        # Stop digging if we go too deep (keeps the output readable)
        if level > max_depth:
            continue
            
        indent = ' ' * 4 * level
        folder_name = os.path.basename(root)
        if folder_name == '':
            folder_name = startpath
            
        print(f"{indent}📂 {folder_name}/")
        
        subindent = ' ' * 4 * (level + 1)
        
        # Print the first 3 files in this folder so we can see what type of data is here
        for f in files[:3]:
            print(f"{subindent}📄 {f}")
            
        if len(files) > 3:
            print(f"{subindent}... and {len(files)-3} more files")

print_directory_tree('/kaggle/input', max_depth=4)

🗺️ Directory tree for: /kaggle/input
📂 input/
    📂 datasets/
        📂 gokulraj324/
            📂 anti-fabrication-toolkit/
                📄 harness_core.py
        📂 gokulrocky/
            📂 final-om-evlautation-upload/
                📄 Kaggle_ChakraTransformer_Evaluation.ipynb
                📂 weights/
                    📄 chakra_transformer_best.pth
                    📄 best.pt
                📂 src/
                    📄 hybrid_refine.py
                    📄 train_yolo.py
                    📄 metrics_engine_v2.py
                    ... and 32 more files
            📂 polypdb-polyp-raw/
                📂 PolypDB/
            📂 chakramodel-kaggle-code/
                📄 requirements.txt
                📂 src/
                    📄 hybrid_refine.py
                    📄 train_yolo.py
                    📄 train_pranet.py
                    ... and 31 more files


In [4]:
%%writefile /kaggle/working/verify_kaggle.py
import os
import sys
import glob
import cv2
import numpy as np
import torch
from pathlib import Path
import hashlib

def md5(fname):
    hash_md5 = hashlib.md5()
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def verify_strict():
    print("=" * 60)
    print("STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION")
    print("=" * 60)
    
    # Identify codebase path
    code_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'chakranet_segmenter.py' in files and 'src' in root:
            code_dir = root
            break
    if not code_dir:
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'chakranet_segmenter.py' in files:
                code_dir = root
                break
    assert code_dir, "FATAL: ChakraModel codebase not found in /kaggle/input"
    sys.path.insert(0, code_dir)
    print(f"[VERIFY] Codebase found at: {code_dir}")
    
    from ultralytics import YOLO
    from chakranet_segmenter import ChakraNet
    from metrics_engine_v2 import MetricsEngineV2

    # BUG 3 FIX: Strict Weight Selection
    yolo_candidates = glob.glob('/kaggle/input/**/best.pt', recursive=True)
    vit_candidates = glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True)
    
    # Strictly filter yolo_candidates (ignore anything with 'transformer' in the path)
    yolo_candidates = [p for p in yolo_candidates if 'transformer' not in p.lower()]
    
    print(f"YOLO candidates found: {yolo_candidates}")
    print(f"ViT candidates found: {vit_candidates}")
    
    assert len(yolo_candidates) == 1, f"FATAL: Expected exactly 1 YOLO weight best.pt, found {len(yolo_candidates)}!"
    assert len(vit_candidates) == 1, f"FATAL: Expected exactly 1 ViT weight chakra_transformer_best.pth, found {len(vit_candidates)}!"
    
    yolo_path = yolo_candidates[0]
    weight_path = vit_candidates[0]

    print(f"Segmenter Checkpoint: {weight_path}")
    print(f"Segmenter MD5: {md5(weight_path)}")
    print(f"YOLO Checkpoint: {yolo_path}")
    print(f"YOLO MD5: {md5(yolo_path)}")
    print("=" * 60)
    
    nonce = os.environ.get("VERIFY_NONCE", "NO_NONCE")
    print(f"VERIFY_NONCE: {nonce}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Executing on device: {device}")
    
    yolo_model = YOLO(yolo_path)
    yolo_model.to(device)
    
    segmenter = ChakraNet(img_size=(384, 384), device=device)
    segmenter.model.to(device)
    
    # Load weights correctly by explicitly stripping 'module.'
    sd = torch.load(weight_path, map_location=device)
    sd_fixed = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    res = segmenter.model.load_state_dict(sd_fixed, strict=False)
    print(f"ChakraNet Weights Loaded. Missing keys: {len(res.missing_keys)}, Unexpected keys: {len(res.unexpected_keys)}")
    
    total_keys = len(segmenter.model.state_dict())
    if len(res.missing_keys) > (0.1 * total_keys):
        print(f"[FATAL] Major weight mismatch! Missing keys > 10% ({len(res.missing_keys)} missing out of {total_keys}).")
        sys.exit(1)
    
    metrics_engine = MetricsEngineV2()
    
    dataset_name = os.environ.get("VERIFY_DATASET_NAME")
    dataset_root = os.environ.get("VERIFY_DATASET_ROOT")
    
    if not dataset_name or not dataset_root:
        print("[FATAL] VERIFY_DATASET_NAME or VERIFY_DATASET_ROOT environment variable not set.")
        sys.exit(1)
        
    print(f"\n============================================================")
    print(f"VERIFYING DATASET: {dataset_name}")
    print(f"============================================================")
    
    d_root = Path(dataset_root)
    images_dir = d_root / "images"
    masks_dir = d_root / "masks"
    
    if not images_dir.exists():
        print(f"[FATAL] Images directory not found: {images_dir}")
        sys.exit(1)
        
    image_paths = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.tif")))
    print(f"Total images found: {len(image_paths)}")
    
    results = {k: [] for k in ["Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"]}
    import csv
    csv_file = "/kaggle/working/per_image_metrics_log.csv"
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Filename", "Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"])
        
    for i, img_path in enumerate(image_paths):
        mask_path = None
        for ext in [".png", ".jpg", ".tif", ".bmp"]:
            potential_path = masks_dir / (img_path.stem + ext)
            if potential_path.exists():
                mask_path = potential_path
                break
                
        if not mask_path:
            print(f"[WARN] Mask not found for {img_path.name}")
            continue
            
        img_bgr = cv2.imread(str(img_path))
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)
        
        img_resized = cv2.resize(img_bgr, (448, 448))
        yolo_results = yolo_model(img_resized, verbose=False)[0]
        
        # Soft prob map initialized to 0
        c_prob = np.zeros((448, 448), dtype=np.float32)
        
        if len(yolo_results.boxes) > 0:
            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)
            box = boxes[0].xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            
            w_box, h_box = x2 - x1, y2 - y1
            px, py = int(0.25 * w_box), int(0.25 * h_box)
            px1, py1 = max(0, x1 - px), max(0, y1 - py)
            px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))
            
            if px2 > px1 and py2 > py1:
                roi_crop = img_resized[py1:py2, px1:px2]
                
                # BUG 2 FIX: Direct manual inference EXACTLY matching chakranet_segmenter.py
                from utils.transforms import letterbox_pad, unletterbox
                img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)
                img_resized_crop, meta = letterbox_pad(img_rgb, target_size=(384, 384))
                
                img_norm = img_resized_crop.astype(np.float32) / 255.0
                mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
                std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
                img_norm = (img_norm - mean) / std
                img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)
                
                try:
                    with torch.no_grad():
                        logits = segmenter.model(img_tensor)
                        prob = torch.sigmoid(logits)
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print("[WARN] Switched to CPU due to CUDA OutOfMemory!")
                        img_tensor_cpu = img_tensor.cpu()
                        segmenter.model.to('cpu')
                        with torch.no_grad():
                            logits = segmenter.model(img_tensor_cpu)
                            prob = torch.sigmoid(logits)
                        segmenter.model.to(device)
                    else:
                        raise
                        
                prob_np = prob.squeeze().cpu().numpy()
                prob_resized = unletterbox(prob_np, meta)
                
                c_prob[py1:py2, px1:px2] = prob_resized
        
        # Explicit binarization ONLY for the ground truth here (network outputs soft prob)
        gt_bin = (gt_resized > 127).astype(np.uint8)
        
        # Pass continuous probability map (c_prob) directly to compute_all 
        # (compute_all handles thresholding for Dice/IoU internally)
        metrics = metrics_engine.compute_all(c_prob, gt_bin)
        
        for k in results.keys():
            results[k].append(metrics[k])
            
        # BUG 4 FIX: Checkpoint saving per image
        with open(csv_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([img_path.name, metrics['Dice'], metrics['mIoU'], metrics['wF-measure'], metrics['S-measure'], metrics['E-measure'], metrics['MAE']])
            
        # IMPORTANT: Print per-image dice WITH FILENAME for canary detection
        dice_score = metrics['Dice']
        print(f"File: {img_path.name} | Dice: {dice_score:.4f}")
        
    print(f"\n[DONE] {dataset_name}")
    print(f"Total Evaluated Images: {len(results['Dice'])}")
    for k in results.keys():
        print(f"Final True Average {k}: {float(np.mean(results[k])):.4f}")
    print(f"============================================================")

if __name__ == "__main__":
    verify_strict()



Writing /kaggle/working/verify_kaggle.py


In [5]:
harness_path = glob.glob('/kaggle/input/**/harness_core.py', recursive=True)
if not harness_path:
    harness_path = glob.glob('/kaggle/working/**/harness_core.py', recursive=True)

if not harness_path:
    print("FATAL: harness_core.py not found! Did you attach the Anti-Fabrication Toolkit dataset?")
    sys.exit(1)

toolkit_dir = os.path.dirname(harness_path[0])
sys.path.insert(0, toolkit_dir)

from harness_core import plant_multi_canary, run_live, verify, sign_verdict

print("Planting canaries...")
secret = plant_multi_canary(WORKING_DATASET)

extra_env = {
    "VERIFY_NONCE": secret["nonce"],
    "VERIFY_DATASET_NAME": VERIFY_DATASET_NAME,
    "VERIFY_DATASET_ROOT": WORKING_DATASET
}

print(f"\nRunning verification for {VERIFY_DATASET_NAME}...")
stdout_text, arrivals, elapsed, retcode = run_live('/kaggle/working/verify_kaggle.py', extra_env, timeout_sec=10000)

print(stdout_text)

if retcode != 0:
    print(f"FATAL: verify_kaggle.py exited with code {retcode}")
else:
    # Parse claimed total from stdout
    match = re.search(r"Total Evaluated Images:\s*(\d+)", stdout_text)
    claimed_total = int(match.group(1)) if match else -1
    
    # Use harness core to verify output logic
    real_count = len(os.listdir(os.path.join(WORKING_DATASET, "images")))
    verdict_dict = verify(stdout_text, arrivals, elapsed, secret, claimed_total, real_count)
    
    card = {
        "project": "ChakraModel",
        "dataset": VERIFY_DATASET_NAME,
        "nonce": secret["nonce"],
        "decision": "ADMITTED" if verdict_dict["pass"] else "WITHHELD",
        "findings": verdict_dict["findings"],
        "elapsed_sec": verdict_dict["elapsed_sec"]
    }
    
    signed_card = sign_verdict(card, secret["hmac_key"])
    
    with open('/kaggle/working/completion_card.json', 'w') as f:
        json.dump(signed_card, f, indent=2)
        
    print(f"\n=========================================")
    print(f"VERDICT: {card['decision']}")
    for finding in card['findings']:
        print(f"  - {finding}")
    print(f"Saved to completion_card.json")
    print(f"=========================================")


Planting canaries...

Running verification for PolypDB_BKAI...
STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION
[VERIFY] Codebase found at: /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/src
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
[WARN] HardwareMonitor unavailable: [Errno 30] Read-only file system: '/kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/logs'
YOLO candidates found: ['/kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/best.pt']
ViT candidates found: ['/kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/chakra_transformer_best.pth']
Segmenter Checkpoint: /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/chakra_transformer_best.p